# Решение задачи multilabel text classification

Цель работы — построить воспроизводимое решение для классификации новостных текстов по пяти возможным меткам. В решении используются только предоставленные данные соревнования: обучающая выборка, тестовая выборка и шаблон отправки.

Подход сочетает классические методы обработки текста и transformer-модель. Такая схема сохраняет интерпретируемую часть решения за счёт TF-IDF и линейных моделей, а также использует более глубокие языковые признаки RuBERT. Итоговый результат сохраняется в один файл `sample_submission.csv`.


In [1]:
import os
import re
import gc
import ast
import html
import math
import json
import time
import random
import warnings
import unicodedata
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

SEED = 322

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)

import numpy as np
import pandas as pd

np.random.seed(SEED)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import hamming_loss, precision_recall_fscore_support, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torch.cuda.amp import autocast, GradScaler
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, set_seed
except Exception as e:
    raise RuntimeError(
        "Для финальной версии нужны torch и transformers. "
        "На Kaggle они обычно уже установлены. Ошибка импорта: " + repr(e)
    )

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    set_seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

try:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

def seed_worker(worker_id):
    worker_seed = SEED
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)

def make_torch_generator(seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


Device: cuda
GPU: Tesla T4
GPU memory total GB: 14.56


In [2]:
N_LABELS = 5
N_FOLDS = 3

MODEL_NAME = "DeepPavlov/rubert-base-cased"

MAX_LEN = 256
TRAIN_BATCH_SIZE = 2
VALID_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
N_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 1

LR_TRANSFORMER = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
MAX_GRAD_NORM = 1.0
DROPOUT = 0.22

USE_AMP = True
USE_GRADIENT_CHECKPOINTING = True
FREEZE_EMBEDDINGS = True
FREEZE_FIRST_N_LAYERS = 1

POS_WEIGHT_POWER = 0.58
POS_WEIGHT_MAX = 5.25
POOLING_MODE = "cls_mean"
CLASSIFIER_HIDDEN = 384

DATALOADER_NUM_WORKERS = 0
PIN_MEMORY = bool(device.type == "cuda")

USE_CLASSICAL = True
USE_TRANSFORMER = True
USE_META_STACKING = True
USE_PRETOKENIZED_DATASETS = True
USE_RATE_GUARD = True
USE_SOURCE_AWARE_GUARD = True
USE_MULTILABEL_RESCUE = True
SAVE_DIAGNOSTICS = False

CHECKPOINT_METRIC = "hamming_loss"
BERT_TEXT_MAX_CHARS = 3800
BERT_TITLE_REPEAT = 2

# Эти коэффициенты введены после прогона с Kaggle loss 0.04176:
# модель была слишком осторожной, all-zero и single-label паттерны были завышены,
# а class_1/class_3/class_4 имели высокий false negative rate.
GLOBAL_RATE_FLOOR_FROM_OOF = np.array([0.96, 0.90, 0.88, 0.90, 0.95], dtype=np.float32)
GLOBAL_RATE_FLOOR_FROM_TRAIN = np.array([0.92, 0.78, 0.82, 0.78, 0.70], dtype=np.float32)
GLOBAL_RATE_CAP_FROM_TRAIN = np.array([1.08, 1.05, 1.18, 1.15, 1.12], dtype=np.float32)
MIN_MULTILABEL_RATE_FROM_TRAIN = 0.72
MAX_MULTILABEL_RESCUE_OOF_DELTA = 0.00055

CONFIG = {
    "SEED": SEED,
    "N_FOLDS": N_FOLDS,
    "MODEL_NAME": MODEL_NAME,
    "MAX_LEN": MAX_LEN,
    "TRAIN_BATCH_SIZE": TRAIN_BATCH_SIZE,
    "VALID_BATCH_SIZE": VALID_BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "EFFECTIVE_TRAIN_BATCH": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "N_EPOCHS": N_EPOCHS,
    "CHECKPOINT_METRIC": CHECKPOINT_METRIC,
    "USE_AMP": USE_AMP,
    "USE_GRADIENT_CHECKPOINTING": USE_GRADIENT_CHECKPOINTING,
    "FREEZE_EMBEDDINGS": FREEZE_EMBEDDINGS,
    "FREEZE_FIRST_N_LAYERS": FREEZE_FIRST_N_LAYERS,
    "POS_WEIGHT_POWER": POS_WEIGHT_POWER,
    "POS_WEIGHT_MAX": POS_WEIGHT_MAX,
    "POOLING_MODE": POOLING_MODE,
    "CLASSIFIER_HIDDEN": CLASSIFIER_HIDDEN,
    "USE_PRETOKENIZED_DATASETS": USE_PRETOKENIZED_DATASETS,
    "USE_META_STACKING": USE_META_STACKING,
    "USE_MULTILABEL_RESCUE": USE_MULTILABEL_RESCUE,
    "BERT_TEXT_MAX_CHARS": BERT_TEXT_MAX_CHARS,
}
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "SEED": 322,
  "N_FOLDS": 3,
  "MODEL_NAME": "DeepPavlov/rubert-base-cased",
  "MAX_LEN": 256,
  "TRAIN_BATCH_SIZE": 2,
  "VALID_BATCH_SIZE": 16,
  "GRAD_ACCUM_STEPS": 4,
  "EFFECTIVE_TRAIN_BATCH": 8,
  "N_EPOCHS": 4,
  "CHECKPOINT_METRIC": "hamming_loss",
  "USE_AMP": true,
  "USE_GRADIENT_CHECKPOINTING": true,
  "FREEZE_EMBEDDINGS": true,
  "FREEZE_FIRST_N_LAYERS": 1,
  "POS_WEIGHT_POWER": 0.58,
  "POS_WEIGHT_MAX": 5.25,
  "POOLING_MODE": "cls_mean",
  "CLASSIFIER_HIDDEN": 384,
  "USE_PRETOKENIZED_DATASETS": true,
  "USE_META_STACKING": true,
  "USE_MULTILABEL_RESCUE": true,
  "BERT_TEXT_MAX_CHARS": 3800
}


In [3]:
RUN_STARTED_AT = time.time()
RUN_LOGS = []

def log_event(message, **kwargs):
    elapsed = time.time() - RUN_STARTED_AT
    row = {"elapsed_min": round(elapsed / 60, 3), "message": message}
    row.update(kwargs)
    RUN_LOGS.append(row)
    suffix = ""
    if kwargs:
        suffix = " | " + ", ".join(f"{k}={v}" for k, v in kwargs.items())
    print(f"[{elapsed/60:7.2f} min] {message}{suffix}")

def gpu_memory(prefix=""):
    if not torch.cuda.is_available():
        return {"allocated_gb": 0.0, "reserved_gb": 0.0}
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"{prefix}GPU memory: allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")
    return {"allocated_gb": round(allocated, 3), "reserved_gb": round(reserved, 3)}

def save_run_logs(path="run_logs.csv"):
    if RUN_LOGS:
        pd.DataFrame(RUN_LOGS).to_csv(path, index=False)
        print(f"Saved {path}")

log_event("Notebook started")
gpu_memory("Initial ")


[   0.00 min] Notebook started
Initial GPU memory: allocated=0.00 GB, reserved=0.00 GB


{'allocated_gb': 0.0, 'reserved_gb': 0.0}

## 1. Чтение данных

На этом этапе загружаются обучающая выборка, тестовая выборка и шаблон отправки. Дополнительно тестовая выборка приводится к порядку `id` из sample submission, чтобы итоговый файл точно совпадал с требуемым форматом соревнования.


In [4]:
def find_file(candidates, required=True):
    roots = [Path("."), Path("/kaggle/working"), Path("/kaggle/input"), Path("/mnt/data")]
    for root in roots:
        if not root.exists():
            continue
        for name in candidates:
            direct = root / name
            if direct.exists():
                return direct
        for path in root.rglob("*"):
            if path.is_file() and path.name in candidates:
                return path
    if required:
        raise FileNotFoundError(f"Не найден файл из списка: {candidates}")
    return None

train_path = find_file(["train.csv"])
test_path = find_file(["test.csv"])
sample_path = find_file(["sample_submission.csv", "sample_submission-5.csv", "sample_submission (5).csv"])

print("train:", train_path)
print("test:", test_path)
print("sample:", sample_path)

def smart_read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    df_tab = pd.read_csv(path, sep="\t")
    if len(df_tab.columns) > 1:
        return df_tab
    return pd.read_csv(path)

train = smart_read_table(train_path)
test = smart_read_table(test_path)
sample_submission = pd.read_csv(sample_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample shape:", sample_submission.shape)
print("train columns:", train.columns.tolist())
print("test columns:", test.columns.tolist())
print("sample columns:", sample_submission.columns.tolist())

log_event("Files loaded", train_rows=len(train), test_rows=len(test), sample_rows=len(sample_submission))


train: /kaggle/input/datasets/anastasiiafirsova/dl00011/train.csv
test: /kaggle/input/datasets/anastasiiafirsova/dl00011/test.csv
sample: /kaggle/input/datasets/anastasiiafirsova/dl00011/sample_submission-5.csv
train shape: (16701, 6)
test shape: (4969, 5)
sample shape: (4969, 2)
train columns: ['id', 'source', 'title', 'text', 'publication_date', 'target']
test columns: ['id', 'source', 'title', 'text', 'publication_date']
sample columns: ['id', 'target']
[   0.04 min] Files loaded | train_rows=16701, test_rows=4969, sample_rows=4969


## 2. Первичный анализ данных и целевой переменной

Здесь проверяется структура таблиц и разбирается целевая переменная. Для multilabel-задачи важно заранее посмотреть частоты классов, долю объектов без меток и распределение источников: эти признаки помогают понять дисбаланс классов и выбрать аккуратную стратегию декодирования предсказаний.


In [5]:
required_train_cols = {"id", "source", "title", "text", "publication_date", "target"}
required_test_cols = {"id", "source", "title", "text", "publication_date"}

missing_train = required_train_cols - set(train.columns)
missing_test = required_test_cols - set(test.columns)

if missing_train:
    raise ValueError(f"В train не хватает столбцов: {missing_train}")
if missing_test:
    raise ValueError(f"В test не хватает столбцов: {missing_test}")

def parse_target(x):
    if isinstance(x, list):
        arr = x
    else:
        arr = ast.literal_eval(str(x))
    arr = [int(v) for v in arr]
    if len(arr) != N_LABELS:
        raise ValueError(f"Некорректная длина target: {arr}")
    return arr

Y = np.array(train["target"].apply(parse_target).tolist(), dtype=np.int64)

print("Y shape:", Y.shape)
print("Positive counts:", Y.sum(axis=0).tolist())
print("Positive rates:", np.round(Y.mean(axis=0), 5).tolist())
print("All-zero rate:", round(float((Y.sum(axis=1) == 0).mean()), 5))

train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

if "id" not in sample_submission.columns:
    raise ValueError("В sample_submission нет столбца id")

test = sample_submission[["id"]].merge(test, on="id", how="left")
if test[["source", "title", "text", "publication_date"]].isna().all(axis=1).any():
    bad = test.loc[test[["source", "title", "text", "publication_date"]].isna().all(axis=1), "id"].head().tolist()
    raise ValueError(f"Не все id из sample_submission найдены в test.csv. Примеры: {bad}")

print("test reordered shape:", test.shape)

Y shape: (16701, 5)
Positive counts: [7145, 2276, 1836, 1243, 544]
Positive rates: [0.42782, 0.13628, 0.10993, 0.07443, 0.03257]
All-zero rate: 0.33788
test reordered shape: (4969, 5)


In [6]:
eda_rows = []
for j in range(N_LABELS):
    eda_rows.append({
        "class": f"class_{j}",
        "count": int(Y[:, j].sum()),
        "rate": float(Y[:, j].mean()),
    })
display(pd.DataFrame(eda_rows))

display(train["source"].value_counts(dropna=False).to_frame("count"))

source_label_rates = train.assign(**{f"class_{j}": Y[:, j] for j in range(N_LABELS)}).groupby("source")[[f"class_{j}" for j in range(N_LABELS)]].mean()
display(source_label_rates)

label_count_distribution = pd.Series(Y.sum(axis=1)).value_counts().sort_index().to_frame("rows")
display(label_count_distribution)

,class,count,rate
0,class_0,7145,0.427819
1,class_1,2276,0.136279
2,class_2,1836,0.109934
3,class_3,1243,0.074427
4,class_4,544,0.032573


,count
source,
Novosti,12759
Svezhesti,3942


,class_0,class_1,class_2,class_3,class_4
source,,,,,
Novosti,0.431382,0.136296,0.109413,0.080884,0.013794
Svezhesti,0.416286,0.136225,0.111618,0.053526,0.093354


,rows
0,5643
1,9209
2,1714
3,133
4,2


In [7]:
ALLOWED_SYMBOL_NOISE = set("№%$€₽+-")

def contains_real_emoji_or_bad_symbol(x, limit=3000):
    for ch in str(x)[:limit]:
        cat = unicodedata.category(ch)
        if ch in ALLOWED_SYMBOL_NOISE:
            continue
        if cat in {"So", "Cs"}:
            return True
    return False

def noise_marker_counts(s):
    s = s.fillna("").astype(str)
    return pd.Series({
        "html_tags": int(s.str.contains(r"<[^>]+>", regex=True).sum()),
        "html_entities": int(s.str.contains(r"&[a-zA-Z0-9#]+;", regex=True).sum()),
        "urls": int(s.str.contains(r"https?://|www\.", regex=True, case=False).sum()),
        "bare_domains": int(s.str.contains(r"\b[a-zA-Z0-9][a-zA-Z0-9_.-]+\.(?:ru|com|net|org|info|io|su|рф)\b", regex=True, case=False).sum()),
        "cdata": int(s.str.contains(r"CDATA", regex=True).sum()),
        "wordpress_blocks": int(s.str.contains(r"wp:|blockquote|figcaption|caption", regex=True, case=False).sum()),
        "agency_boilerplate": int(s.str.contains(r"РИА Новости|ТАСС|Интерфакс|Reuters|Bloomberg|Источник|Фото|Видео", regex=True, case=False).sum()),
        "real_emoji_or_bad_symbol_first_3000_chars": int(s.apply(contains_real_emoji_or_bad_symbol).sum()),
    })

def show_high_noise_examples(df, name, n=5):
    text = df["text"].fillna("").astype(str)
    score = (
        text.str.count(r"<[^>]+>") +
        text.str.count(r"&[a-zA-Z0-9#]+;") +
        text.str.count(r"https?://|www\.") * 3 +
        text.str.count(r"CDATA") * 2
    )
    ex = df.loc[score.sort_values(ascending=False).head(n).index, ["id", "source", "title", "text"]].copy()
    ex["noise_score"] = score.loc[ex.index].values
    ex["text"] = ex["text"].astype(str).str[:900]
    print(f"High-noise examples: {name}")
    display(ex)
    if SAVE_DIAGNOSTICS:
        ex.to_csv(f"diagnostics_high_noise_examples_{name}.csv", index=False)

print("Missing values train:")
display(train.isna().sum().to_frame("missing"))
print("Missing values test:")
display(test.isna().sum().to_frame("missing"))

print("Noise markers before cleaning, train text:")
display(noise_marker_counts(train["text"]).to_frame("count"))
print("Noise markers before cleaning, test text:")
display(noise_marker_counts(test["text"]).to_frame("count"))

show_high_noise_examples(train, "train")
show_high_noise_examples(test, "test")

for name, df in [("train", train), ("test", test)]:
    dt = pd.to_datetime(df["publication_date"], errors="coerce")
    print(name, "date range:", dt.min(), "—", dt.max(), "missing dates:", int(dt.isna().sum()))
    print(name, "source distribution:")
    display(df["source"].value_counts(dropna=False).to_frame("count"))

log_event("Raw data diagnostics completed")

Missing values train:


,missing
id,0
source,0
title,0
text,0
publication_date,0
target,0


Missing values test:


,missing
id,0
source,0
title,0
text,0
publication_date,0


Noise markers before cleaning, train text:


...

train date range: 2019-12-23 00:00:00 — 2020-07-30 23:59:00 missing dates: 0
train source distribution:


,count
source,
Novosti,12759
Svezhesti,3942


test date range: 2020-01-01 07:35:00 — 2020-08-31 21:06:00 missing dates: 0
test source distribution:


,count
source,
Novosti,1996
Zholtosti,1538
Spletnesti,867
Svezhesti,568


[   0.20 min] Raw data diagnostics completed


## 3. Очистка и подготовка текста

Тексты в датасете содержат HTML-разметку, служебные фрагменты, ссылки, домены, спецсимволы и шумовые вставки. Очистка нужна, чтобы модели обучались на содержании новости, а не на техническом мусоре.

Для классических моделей формируется компактный очищенный текст с source/month-маркерами. Для transformer сохраняется более естественная текстовая форма с заголовком и основным текстом, чтобы языковая модель получала контекст, близкий к исходной новости.


In [8]:
TAG_WORDS = {
    "content", "source", "span", "div", "p", "article", "blockquote", "section",
    "item", "news", "doc", "text", "author", "time", "body", "html", "meta",
    "strong", "em", "li", "ul", "ol", "hr", "br", "class", "quote", "style",
    "script", "href", "src", "wp", "paragraph", "figure", "figcaption",
    "iframe", "noscript", "amp", "nbsp", "laquo", "raquo", "mdash",
    "align", "width", "height", "target", "rel", "nofollow", "noopener",
    "telegram", "instagram", "facebook", "twitter", "youtube", "vk", "tme"
}

KEEP_SYMBOLS = set("%$€₽№+-.,:;!?()«»\"'/")

SOURCE_AGENCY_WORDS = [
    "риа новости", "тасс", "интерфакс", "рбк", "коммерсантъ", "ведомости",
    "известия", "лента.ру", "газета.ру", "reuters", "bloomberg", "associated press"
]

BOILERPLATE_PATTERNS = [
    r"\bподписывайтесь на [^.?!]{0,120}",
    r"\bподписаться на [^.?!]{0,120}",
    r"\bчитайте также[:：]?",
    r"\bсмотрите также[:：]?",
    r"\bпо теме[:：]?",
    r"\bподелиться[:：]?",
    r"\bкомментарии[:：]?",
    r"\bпохожие материалы[:：]?",
    r"\bреклама[:：]?",
    r"\bфото[:：]?\s*[^.?!]{0,80}",
    r"\bвидео[:：]?\s*[^.?!]{0,80}",
    r"\bисточник[:：]?\s*[^.?!]{0,100}",
    r"\bматериал подготовлен[^.?!]{0,160}",
    r"\bпо материалам [^.?!]{0,160}",
    r"\bпри полном или частичном использовании материалов[^.?!]{0,200}",
    r"\bвсе права защищены[^.?!]{0,120}",
    r"\bнашли ошибку[^.?!]{0,120}",
]

def normalize_source_name(x):
    x = str(x).strip().lower()
    x = re.sub(r"\W+", "_", x, flags=re.U).strip("_")
    if x in {"novosti", "svezhesti"}:
        return x
    return "other"

def remove_bad_unicode(text):
    out = []
    for ch in str(text):
        cat = unicodedata.category(ch)
        if cat in {"Cf", "Co", "Cs"}:
            out.append(" ")
        elif cat.startswith("S") and ch not in KEEP_SYMBOLS:
            out.append(" ")
        else:
            out.append(ch)
    return "".join(out)

def html_unescape_deep(text, n=4):
    text = str(text)
    for _ in range(n):
        new_text = html.unescape(text)
        if new_text == text:
            break
        text = new_text
    return text

def remove_urls_domains_emails(text):
    text = re.sub(r"[\w\.-]+@[\w\.-]+\.\w+", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text, flags=re.I)
    text = re.sub(r"\b(?:t\.me|vk\.com|facebook\.com|twitter\.com|x\.com|instagram\.com|youtube\.com|youtu\.be|telegram\.me)/\S+", " ", text, flags=re.I)
    text = re.sub(r"\b[a-zA-Z0-9][a-zA-Z0-9_\-\.]{1,80}\.(?:ru|рф|com|net|org|info|io|su|ua|by|kz|рф)\b", " ", text, flags=re.I)
    return text

def remove_markup(text):
    text = re.sub(r"<!\[CDATA\[|\]\]>", " ", text, flags=re.I)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.S)
    text = re.sub(r"<script.*?</script>", " ", text, flags=re.S | re.I)
    text = re.sub(r"<style.*?</style>", " ", text, flags=re.S | re.I)
    text = re.sub(r"\[/?(?:caption|gallery|embed|video|audio)[^\]]*\]", " ", text, flags=re.I)
    text = re.sub(r"!\[[^\]]*\]\([^)]+\)", " ", text)
    text = re.sub(r"\[[^\]]{0,80}\]\([^)]+\)", " ", text)
    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"\{[^{}]{0,200}\}", " ", text)
    return text

def remove_source_boilerplate(text):
    text = str(text)
    # Типовая новостная шапка: "МОСКВА, 15 мая — РИА Новости."
    text = re.sub(
        r"^\s*[А-ЯA-ZЁ][А-ЯA-Zа-яa-zёЁ\- ]{2,35}\s*,?\s*\d{1,2}\s+[а-яА-ЯёЁa-zA-Z]+\s*[-—–]\s*(?:"
        + "|".join(map(re.escape, SOURCE_AGENCY_WORDS)) + r")\.?",
        " ",
        text,
        flags=re.I,
    )
    # Удаляем явные подписи источников, но не смысловые слова вокруг.
    text = re.sub(r"\b(?:сообщает|передает|пишет|отмечает|уточняет)\s+(?:" + "|".join(map(re.escape, SOURCE_AGENCY_WORDS)) + r")\b", " ", text, flags=re.I)
    text = re.sub(r"\b(?:" + "|".join(map(re.escape, SOURCE_AGENCY_WORDS)) + r")\b", " ", text, flags=re.I)
    return text

def normalize_numbers_and_punctuation(text):
    text = re.sub(r"(?<=\d),(?=\d)", ".", text)
    text = re.sub(r"([!?.,:;]){2,}", r"\1", text)
    text = re.sub(r"[«»„“”]", '"', text)
    text = re.sub(r"[‐‑‒–—―]", "-", text)
    return text

def clean_common(text):
    if pd.isna(text):
        return ""
    text = html_unescape_deep(text)
    text = remove_markup(text)
    text = remove_urls_domains_emails(text)
    text = remove_source_boilerplate(text)
    text = remove_bad_unicode(text)
    text = normalize_numbers_and_punctuation(text)
    text = re.sub(r"[\r\n\t]+", " ", text)

    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.I | re.U)

    text = re.sub(r"&[a-zA-Z0-9#]+;", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = re.sub(r"[|•●▪◆■□▲▼★☆]+", " ", text)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_for_transformer(text):
    text = clean_common(text)
    text = re.sub(r"[^\wа-яА-ЯёЁ0-9%$€₽№+\-.,:;!?()«»\"'/ ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_for_tfidf(text):
    text = clean_common(text).lower().replace("ё", "е")
    text = re.sub(r"[^0-9a-zа-я%$€₽№+\-.,:/ ]+", " ", text)
    tokens = []
    prev = None
    repeat_count = 0
    for t in text.split():
        t = t.strip(".,:;!?()[]{}«»\"'")
        if not t or t in TAG_WORDS:
            continue
        if len(t) == 1 and not t.isdigit():
            continue
        if t == prev:
            repeat_count += 1
            if repeat_count >= 2:
                continue
        else:
            repeat_count = 0
        tokens.append(t)
        prev = t
    text = " ".join(tokens)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def smart_trim(text, max_chars=BERT_TEXT_MAX_CHARS, head_ratio=0.80):
    text = str(text)
    if len(text) <= max_chars:
        return text
    head = int(max_chars * head_ratio)
    tail = max_chars - head
    return text[:head].rstrip() + " ... " + text[-tail:].lstrip()

def make_source_token(source):
    group = normalize_source_name(source)
    if group == "novosti":
        return " source_novosti"
    if group == "svezhesti":
        return " source_svezhesti"
    return ""

def add_text_features(df):
    df = df.copy()
    df["source"] = df["source"].fillna("unknown").astype(str)
    df["title"] = df["title"].fillna("").astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    df["publication_date"] = df["publication_date"].fillna("").astype(str)

    dt = pd.to_datetime(df["publication_date"], errors="coerce")
    df["month_token"] = "month_" + dt.dt.month.fillna(0).astype(int).astype(str)
    df["source_group"] = df["source"].map(normalize_source_name)

    title_tfidf = df["title"].apply(clean_for_tfidf)
    text_tfidf = df["text"].apply(clean_for_tfidf)

    title_bert = df["title"].apply(clean_for_transformer)
    text_bert = df["text"].apply(clean_for_transformer).apply(smart_trim)

    source_token = df["source"].apply(make_source_token)
    date_token = " " + df["month_token"]

    title_for_tfidf = (title_tfidf + " ") * 3
    df["tfidf_text"] = (
        source_token + date_token + " " +
        title_for_tfidf + " " + text_tfidf
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    title_for_bert = ((title_bert + " [SEP] ") * BERT_TITLE_REPEAT).str.strip()
    df["bert_text"] = (
        "Заголовок: " + title_for_bert + " Текст: " + text_bert
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    empty_tfidf = df["tfidf_text"].str.len() == 0
    df.loc[empty_tfidf, "tfidf_text"] = title_tfidf[empty_tfidf]

    empty_bert = df["bert_text"].str.len() == 0
    df.loc[empty_bert, "bert_text"] = title_bert[empty_bert]

    df["clean_title_len"] = title_tfidf.str.len().astype(np.int32)
    df["clean_text_len"] = text_tfidf.str.len().astype(np.int32)
    df["bert_text_len"] = df["bert_text"].str.len().astype(np.int32)
    df["tfidf_text_len"] = df["tfidf_text"].str.len().astype(np.int32)

    return df

log_event("Text cleaning started")
clean_t0 = time.time()
train = add_text_features(train)
test = add_text_features(test)
log_event("Text cleaning finished", seconds=round(time.time() - clean_t0, 2))

print(train[["tfidf_text", "bert_text"]].head(2).to_string())
print("Empty train tfidf:", int((train["tfidf_text"].str.len() == 0).sum()))
print("Empty test tfidf:", int((test["tfidf_text"].str.len() == 0).sum()))

length_report = pd.DataFrame({
    "dataset": ["train", "test"],
    "tfidf_len_mean": [train["tfidf_text"].str.len().mean(), test["tfidf_text"].str.len().mean()],
    "tfidf_len_p50": [train["tfidf_text"].str.len().median(), test["tfidf_text"].str.len().median()],
    "tfidf_len_p95": [train["tfidf_text"].str.len().quantile(0.95), test["tfidf_text"].str.len().quantile(0.95)],
    "bert_len_mean": [train["bert_text"].str.len().mean(), test["bert_text"].str.len().mean()],
    "bert_len_p50": [train["bert_text"].str.len().median(), test["bert_text"].str.len().median()],
    "bert_len_p95": [train["bert_text"].str.len().quantile(0.95), test["bert_text"].str.len().quantile(0.95)],
})
print("Length report after cleaning:")
display(length_report)

print("Noise markers after cleaning, train tfidf_text:")
display(noise_marker_counts(train["tfidf_text"]).to_frame("count"))
print("Noise markers after cleaning, test tfidf_text:")
display(noise_marker_counts(test["tfidf_text"]).to_frame("count"))

def cleaned_examples(df, name, n=6):
    ex = df[["id", "source", "title", "tfidf_text", "bert_text"]].copy()
    ex["tfidf_len"] = ex["tfidf_text"].str.len()
    ex["bert_len"] = ex["bert_text"].str.len()
    ex = ex.sort_values("tfidf_len", ascending=False).head(n)
    ex["tfidf_text"] = ex["tfidf_text"].str[:700]
    ex["bert_text"] = ex["bert_text"].str[:700]
    print(f"Cleaned long examples: {name}")
    display(ex)
    if SAVE_DIAGNOSTICS:
        ex.to_csv(f"diagnostics_cleaned_long_examples_{name}.csv", index=False)

cleaned_examples(train, "train")
cleaned_examples(test, "test")

if SAVE_DIAGNOSTICS:
    length_report.to_csv("diagnostics_clean_text_lengths.csv", index=False)


[   0.20 min] Text cleaning started
[   2.11 min] Text cleaning finished | seconds=114.59
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

,dataset,tfidf_len_mean,tfidf_len_p50,tfidf_len_p95,bert_len_mean,bert_len_p50,bert_len_p95
0,train,1678.891444,1392.0,3551.0,1630.569666,1433.0,3724.0
1,test,2093.910244,1330.0,6718.0,1666.749447,1349.0,3970.0


Noise markers after cleaning, train tfidf_text:


,count
html_tags,0
html_entities,0
urls,0
bare_domains,0
cdata,0
wordpress_blocks,27
agency_boilerplate,2843
real_emoji_or_bad_symbol_first_3000_chars,0


Noise markers after cleaning, test tfidf_text:


,count
html_tags,0
html_entities,0
urls,0
bare_domains,0
cdata,0
wordpress_blocks,0
agency_boilerplate,127
real_emoji_or_bad_symbol_first_3000_chars,0


Cleaned long examples: train


,id,source,title,tfidf_text,bert_text,tfidf_len,bert_len
8462,8462,Svezhesti,"«Если что-то ужасное произойдет, хоть кто-то д...",source_svezhesti month_4 если что-то ужасное п...,"Заголовок: ""Если что-то ужасное произойдет, хо...",24498,3948
6403,6403,Svezhesti,Силовой захват,source_svezhesti month_3 силовой захват силово...,Заголовок: Силовой захват [SEP] Силовой захват...,22165,3865
10509,10509,Novosti,Меры борьбы с распространением COVID-19 в России,source_novosti month_5 меры борьбы распростран...,Заголовок: Меры борьбы с распространением COVI...,20757,3933
11029,11029,Svezhesti,«Дети просили не убивать их»,source_svezhesti month_5 дети просили не убива...,"Заголовок: ""Дети просили не убивать их"" [SEP] ...",19371,3892
12199,12199,Svezhesti,«Многим приходится тяжко»,source_svezhesti month_6 многим приходится тяж...,"Заголовок: ""Многим приходится тяжко"" [SEP] ""Мн...",16637,3886
10135,10135,Novosti,Работа Владимира Путина на посту президента Ро...,source_novosti month_5 работа владимира путина...,Заголовок: Работа Владимира Путина на посту пр...,16123,3937


Cleaned long examples: test


,id,source,title,tfidf_text,bert_text,tfidf_len,bert_len
1982,18683,Zholtosti,"«Я бы дорого отдал, чтобы моя жизнь стала преж...",month_6 бы дорого отдал чтобы моя жизнь стала ...,"Заголовок: ""Я бы дорого отдал, чтобы моя жизнь...",42946,3936
1803,18504,Zholtosti,«Советский запах еще не выветрился. Мы по-преж...,month_5 советский запах еще не выветрился мы п...,"Заголовок: ""Советский запах еще не выветрился....",42345,3964
851,17552,Zholtosti,«Успел сказать: извини — и выстрелил»,month_3 успел сказать извини выстрелил успел с...,"Заголовок: ""Успел сказать: извини - и выстрели...",40778,3910
1723,18424,Zholtosti,Всех постарался придушить,month_5 всех постарался придушить всех постара...,Заголовок: Всех постарался придушить [SEP] Все...,36651,3887
1653,18354,Zholtosti,"«Я боюсь контроля, чипирования, ограничения св...",month_5 боюсь контроля чипирования ограничения...,"Заголовок: ""Я боюсь контроля, чипирования, огр...",36166,3979
115,16816,Zholtosti,"«Вас беспокоит Леонид Слуцкий, не депутат»",month_1 вас беспокоит леонид слуцкий не депута...,"Заголовок: ""Вас беспокоит Леонид Слуцкий, не д...",36067,3921


## 4. Разбиение на фолды

Разбиение строится так, чтобы в каждом фолде сохранялись близкие доли классов, источников и количества меток на объект. Это важно для честной OOF-оценки: валидационные фолды должны быть похожи на обучающую выборку и не ломать баланс редких классов.


In [9]:
def make_strat_key(df, y):
    label_key = ["".join(map(str, row.tolist())) for row in y]
    source_key = df["source"].fillna("unknown").astype(str).str[:20].tolist()
    n_key = y.sum(axis=1).astype(str).tolist()
    raw = pd.Series([f"{a}_{b}_{c}" for a, b, c in zip(label_key, source_key, n_key)])
    counts = raw.value_counts()
    key = raw.where(raw.map(counts) >= N_FOLDS, pd.Series([f"{a}_n{c}" for a, c in zip(label_key, n_key)]))
    counts2 = key.value_counts()
    key = key.where(key.map(counts2) >= N_FOLDS, pd.Series(n_key))
    return key.astype(str).values

strat_key = make_strat_key(train, Y)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = np.zeros(len(train), dtype=np.int64)

for fold, (_, va_idx) in enumerate(skf.split(train, strat_key)):
    folds[va_idx] = fold

train["fold"] = folds

for fold in range(N_FOLDS):
    idx = np.where(folds == fold)[0]
    print("fold", fold, "size", len(idx), "rates", np.round(Y[idx].mean(axis=0), 4).tolist(), "all_zero", round(float((Y[idx].sum(axis=1)==0).mean()), 4))

fold 0 size 5567 rates [0.4279, 0.1367, 0.1092, 0.0745, 0.0327] all_zero 0.3379
fold 1 size 5567 rates [0.4273, 0.1362, 0.1103, 0.0745, 0.0325] all_zero 0.3379
fold 2 size 5567 rates [0.4282, 0.136, 0.1103, 0.0742, 0.0325] all_zero 0.3379


In [10]:
fold_rows = []
for fold in range(N_FOLDS):
    idx = np.where(folds == fold)[0]
    row = {"fold": fold, "rows": len(idx), "all_zero_rate": float((Y[idx].sum(axis=1) == 0).mean())}
    for j in range(N_LABELS):
        row[f"class_{j}_rate"] = float(Y[idx, j].mean())
    fold_rows.append(row)
fold_report = pd.DataFrame(fold_rows)
print("Fold balance report:")
display(fold_report)
if SAVE_DIAGNOSTICS:
    fold_report.to_csv("diagnostics_fold_balance.csv", index=False)
log_event("Folds prepared")


Fold balance report:


,fold,rows,all_zero_rate,class_0_rate,class_1_rate,class_2_rate,class_3_rate,class_4_rate
0,0,5567,0.337884,0.427879,0.136698,0.109215,0.074546,0.032693
1,1,5567,0.337884,0.427340,0.136160,0.110293,0.074546,0.032513
2,2,5567,0.337884,0.428238,0.135980,0.110293,0.074187,0.032513


[   2.35 min] Folds prepared


## 5. Классические текстовые модели

Классическая часть решения использует TF-IDF-признаки по словам и символам. Word-level признаки помогают ловить смысловые термины, а char-level признаки устойчивее к шуму, разным формам слов и фрагментам грязного текста.

На этих признаках обучаются несколько линейных моделей, после чего их вероятности смешиваются. Этот блок выполняет требование соревнования о наличии методов классического ML и даёт независимый источник предсказаний для ансамбля.


In [11]:
def fit_predict_classical(train_texts, valid_texts, test_texts, y_train):
    t0 = time.time()
    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.985,
        max_features=170000,
        sublinear_tf=True,
        dtype=np.float32
    )
    char_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 6),
        min_df=2,
        max_df=0.995,
        max_features=110000,
        sublinear_tf=True,
        dtype=np.float32
    )

    X_tr_word = word_vec.fit_transform(train_texts)
    X_va_word = word_vec.transform(valid_texts)
    X_te_word = word_vec.transform(test_texts)

    X_tr_char = char_vec.fit_transform(train_texts)
    X_va_char = char_vec.transform(valid_texts)
    X_te_char = char_vec.transform(test_texts)

    X_tr = hstack([X_tr_word, X_tr_char]).tocsr()
    X_va = hstack([X_va_word, X_va_char]).tocsr()
    X_te = hstack([X_te_word, X_te_char]).tocsr()

    print("TF-IDF shapes:", X_tr.shape, X_va.shape, X_te.shape)

    models = [
        ("lr_balanced", OneVsRestClassifier(LogisticRegression(
            C=4.0,
            solver="liblinear",
            class_weight="balanced",
            max_iter=1400,
            random_state=SEED
        ), n_jobs=-1)),
        ("lr_plain", OneVsRestClassifier(LogisticRegression(
            C=2.0,
            solver="liblinear",
            class_weight=None,
            max_iter=1400,
            random_state=SEED
        ), n_jobs=-1)),
        ("sgd", OneVsRestClassifier(SGDClassifier(
            loss="modified_huber",
            alpha=1.2e-5,
            penalty="l2",
            max_iter=2800,
            tol=1e-4,
            class_weight="balanced",
            random_state=SEED
        ), n_jobs=-1)),
        ("cnb", OneVsRestClassifier(ComplementNB(alpha=0.30), n_jobs=-1))
    ]

    va_parts = []
    te_parts = []

    for name, model in models:
        mt0 = time.time()
        model.fit(X_tr, y_train)
        va_p = model.predict_proba(X_va)
        te_p = model.predict_proba(X_te)
        va_parts.append(np.clip(va_p, 0, 1))
        te_parts.append(np.clip(te_p, 0, 1))
        print(f"classical model done: {name}, seconds={time.time() - mt0:.1f}")

    va_pred = 0.42 * va_parts[0] + 0.18 * va_parts[1] + 0.28 * va_parts[2] + 0.12 * va_parts[3]
    te_pred = 0.42 * te_parts[0] + 0.18 * te_parts[1] + 0.28 * te_parts[2] + 0.12 * te_parts[3]

    del X_tr_word, X_va_word, X_te_word, X_tr_char, X_va_char, X_te_char, X_tr, X_va, X_te
    gc.collect()

    print(f"classical fold total seconds={time.time() - t0:.1f}")
    return np.clip(va_pred, 0, 1), np.clip(te_pred, 0, 1)

classical_oof = np.zeros((len(train), N_LABELS), dtype=np.float32)
classical_test = np.zeros((len(test), N_LABELS), dtype=np.float32)

if USE_CLASSICAL:
    log_event("Classical OOF training started")
    for fold in range(N_FOLDS):
        print("\nCLASSICAL FOLD", fold)
        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]

        va_pred, te_pred = fit_predict_classical(
            train.loc[tr_idx, "tfidf_text"].values,
            train.loc[va_idx, "tfidf_text"].values,
            test["tfidf_text"].values,
            Y[tr_idx]
        )

        classical_oof[va_idx] = va_pred
        classical_test += te_pred / N_FOLDS
        fold_hl_05 = hamming_loss(Y[va_idx], (va_pred >= 0.5).astype(int))
        log_event("Classical fold completed", fold=fold, hamming_at_05=round(fold_hl_05, 5))

    print("Classical OOF hamming at 0.5:", hamming_loss(Y, (classical_oof >= 0.5).astype(int)))
    classical_rate_report = pd.DataFrame({
        "class": [f"class_{j}" for j in range(N_LABELS)],
        "true_rate": Y.mean(axis=0),
        "classical_prob_mean": classical_oof.mean(axis=0),
        "classical_pred_rate_05": (classical_oof >= 0.5).mean(axis=0),
    })
    display(classical_rate_report)
    if SAVE_DIAGNOSTICS:
        classical_rate_report.to_csv("diagnostics_classical_oof_rates.csv", index=False)
else:
    classical_oof[:] = 0.0
    classical_test[:] = 0.0


[   2.35 min] Classical OOF training started

CLASSICAL FOLD 0
TF-IDF shapes: (11134, 280000) (5567, 280000) (4969, 280000)
classical model done: lr_balanced, seconds=36.1
classical model done: lr_plain, seconds=22.8
classical model done: sgd, seconds=4.5
classical model done: cnb, seconds=2.3
classical fold total seconds=154.5
[   4.92 min] Classical fold completed | fold=0, hamming_at_05=0.05856

CLASSICAL FOLD 1
TF-IDF shapes: (11134, 280000) (5567, 280000) (4969, 280000)
classical model done: lr_balanced, seconds=32.2
classical model done: lr_plain, seconds=21.8
classical model done: sgd, seconds=3.7
classical model done: cnb, seconds=2.3
classical fold total seconds=145.2
[   7.34 min] Classical fold completed | fold=1, hamming_at_05=0.05673

CLASSICAL FOLD 2
TF-IDF shapes: (11134, 280000) (5567, 280000) (4969, 280000)
classical model done: lr_balanced, seconds=32.2
classical model done: lr_plain, seconds=21.5
classical model done: sgd, seconds=3.6
classical model done: cnb, secon

,class,true_rate,classical_prob_mean,classical_pred_rate_05
0,class_0,0.427819,0.426509,0.419915
1,class_1,0.136279,0.148616,0.127597
2,class_2,0.109934,0.128367,0.104126
3,class_3,0.074427,0.084668,0.062990
4,class_4,0.032573,0.036962,0.021615


## 6. Transformer-модель

Transformer-блок использует RuBERT для извлечения контекстных признаков из текста. Модель обучается по фолдам и даёт OOF-предсказания для train, а также усреднённый прогноз для test.

Этот компонент нужен, чтобы дополнить TF-IDF: классические модели хорошо работают с поверхностными текстовыми паттернами, а RuBERT лучше учитывает контекст и смысловые связи внутри новости.


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EncodedTextDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels=None):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

def encode_texts(texts, name, batch_size=512):
    log_event(f"Tokenization started: {name}", rows=len(texts))
    t0 = time.time()
    enc = tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_attention_mask=True,
        return_tensors=None,
        batch_size=batch_size,
    )
    input_ids = np.asarray(enc["input_ids"], dtype=np.int32)
    attention_mask = np.asarray(enc["attention_mask"], dtype=np.int8)
    log_event(f"Tokenization finished: {name}", seconds=round(time.time() - t0, 2))
    return {"input_ids": input_ids, "attention_mask": attention_mask}

if USE_TRANSFORMER and USE_PRETOKENIZED_DATASETS:
    train_encoded = encode_texts(train["bert_text"].values, "train")
    test_encoded = encode_texts(test["bert_text"].values, "test")
else:
    train_encoded = None
    test_encoded = None

class RubertMultilabelModel(nn.Module):
    def __init__(self, model_name, n_labels, pos_weight=None, init_bias=None):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)

        if USE_GRADIENT_CHECKPOINTING and hasattr(self.backbone, "gradient_checkpointing_enable"):
            self.backbone.gradient_checkpointing_enable()

        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(DROPOUT)

        if POOLING_MODE == "cls_mean":
            self.proj = nn.Sequential(
                nn.Linear(hidden * 2, CLASSIFIER_HIDDEN),
                nn.LayerNorm(CLASSIFIER_HIDDEN),
                nn.GELU(),
                nn.Dropout(DROPOUT),
            )
            self.classifier = nn.Linear(CLASSIFIER_HIDDEN, n_labels)
        else:
            self.proj = nn.Identity()
            self.classifier = nn.Linear(hidden, n_labels)

        if init_bias is not None:
            with torch.no_grad():
                self.classifier.bias.copy_(torch.tensor(init_bias, dtype=torch.float32))

        if pos_weight is None:
            pos_weight = np.ones(n_labels, dtype=np.float32)
        self.register_buffer("pos_weight", torch.tensor(pos_weight, dtype=torch.float32))

        self.apply_freezing()

    def apply_freezing(self):
        if FREEZE_EMBEDDINGS and hasattr(self.backbone, "embeddings"):
            for p in self.backbone.embeddings.parameters():
                p.requires_grad = False

        encoder = getattr(self.backbone, "encoder", None)
        if encoder is not None and hasattr(encoder, "layer"):
            for layer in encoder.layer[:FREEZE_FIRST_N_LAYERS]:
                for p in layer.parameters():
                    p.requires_grad = False

    def masked_mean_pooling(self, hidden_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(hidden_states.dtype)
        summed = (hidden_states * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return summed / denom

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]

        if POOLING_MODE == "cls_mean":
            mean_pool = self.masked_mean_pooling(out.last_hidden_state, attention_mask)
            features = torch.cat([cls, mean_pool], dim=1)
            features = self.proj(features)
        else:
            features = cls

        logits = self.classifier(self.dropout(features))

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
            loss = loss_fn(logits, labels)
        return logits, loss

def make_pos_weight(y):
    pos = y.sum(axis=0).astype(np.float32)
    neg = len(y) - pos
    raw = neg / np.maximum(pos, 1.0)
    weights = np.power(raw, POS_WEIGHT_POWER)
    weights = np.clip(weights, 1.0, POS_WEIGHT_MAX)
    print("pos_weight:", np.round(weights, 3).tolist())
    return weights.astype(np.float32)

def make_bias(y):
    rate = y.mean(axis=0)
    rate = np.clip(rate, 1e-4, 1 - 1e-4)
    return np.log(rate / (1 - rate)).astype(np.float32)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def make_encoded_dataset(encoded, indices=None, labels=None):
    if indices is None:
        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]
    else:
        input_ids = encoded["input_ids"][indices]
        attention_mask = encoded["attention_mask"][indices]
    return EncodedTextDataset(input_ids, attention_mask, labels)

def predict_transformer_encoded(model, encoded, indices=None, batch_size=VALID_BATCH_SIZE):
    ds = make_encoded_dataset(encoded, indices=indices, labels=None)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=DATALOADER_NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker
    )
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=PIN_MEMORY)
            attention_mask = batch["attention_mask"].to(device, non_blocking=PIN_MEMORY)
            with autocast(enabled=(USE_AMP and device.type == "cuda")):
                logits, _ = model(input_ids=input_ids, attention_mask=attention_mask)
            preds.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.vstack(preds)

def eval_transformer_loss_encoded(model, encoded, indices, labels, batch_size=VALID_BATCH_SIZE):
    ds = make_encoded_dataset(encoded, indices=indices, labels=labels)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=DATALOADER_NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker
    )
    model.eval()
    losses = []
    preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=PIN_MEMORY)
            attention_mask = batch["attention_mask"].to(device, non_blocking=PIN_MEMORY)
            y = batch["labels"].to(device, non_blocking=PIN_MEMORY)
            with autocast(enabled=(USE_AMP and device.type == "cuda")):
                logits, loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=y)
            losses.append(float(loss.detach().cpu()))
            preds.append(torch.sigmoid(logits).detach().cpu().numpy())
    return float(np.mean(losses)), np.vstack(preds)


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[   9.77 min] Tokenization started: train | rows=16701
[  10.04 min] Tokenization finished: train | seconds=16.27
[  10.05 min] Tokenization started: test | rows=4969
[  10.13 min] Tokenization finished: test | seconds=4.74


In [13]:
def train_transformer_fold(fold, tr_idx, va_idx):
    clear_memory()
    gpu_memory(f"Before transformer fold {fold} ")
    fold_t0 = time.time()

    y_tr = Y[tr_idx]
    y_va = Y[va_idx]

    pos_weight = make_pos_weight(y_tr)
    init_bias = make_bias(y_tr)

    model = RubertMultilabelModel(
        MODEL_NAME,
        N_LABELS,
        pos_weight=pos_weight,
        init_bias=init_bias
    ).to(device)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Transformer fold {fold}: trainable params {trainable_params:,} / {total_params:,}")

    ds_tr = make_encoded_dataset(train_encoded, indices=tr_idx, labels=y_tr)
    train_generator = make_torch_generator(SEED)
    loader_tr = DataLoader(
        ds_tr,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=DATALOADER_NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=train_generator
    )

    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)],
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]

    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR_TRANSFORMER)
    total_steps = math.ceil(len(loader_tr) / GRAD_ACCUM_STEPS) * N_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    scaler = GradScaler(enabled=(USE_AMP and device.type == "cuda"))

    best_loss = float("inf")
    best_metric = float("inf")
    best_state = None
    bad_epochs = 0
    epoch_logs = []

    for epoch in range(1, N_EPOCHS + 1):
        epoch_t0 = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        n_batches = 0

        for step, batch in enumerate(loader_tr, start=1):
            input_ids = batch["input_ids"].to(device, non_blocking=PIN_MEMORY)
            attention_mask = batch["attention_mask"].to(device, non_blocking=PIN_MEMORY)
            y = batch["labels"].to(device, non_blocking=PIN_MEMORY)

            with autocast(enabled=(USE_AMP and device.type == "cuda")):
                _, loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=y)
                loss = loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS
            n_batches += 1

            if step % GRAD_ACCUM_STEPS == 0 or step == len(loader_tr):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        val_loss, val_probs = eval_transformer_loss_encoded(model, train_encoded, va_idx, y_va)
        val_hl_05 = hamming_loss(y_va, (val_probs >= 0.5).astype(int))
        val_rates_05 = (val_probs >= 0.5).mean(axis=0)
        epoch_seconds = time.time() - epoch_t0

        row = {
            "fold": fold,
            "epoch": epoch,
            "train_loss": running_loss / max(n_batches, 1),
            "valid_loss": val_loss,
            "hamming_at_05": val_hl_05,
            "seconds": epoch_seconds,
        }
        for j in range(N_LABELS):
            row[f"pred_rate_05_class_{j}"] = float(val_rates_05[j])
        epoch_logs.append(row)

        print(
            f"fold {fold} epoch {epoch}: "
            f"train_loss={row['train_loss']:.5f}, "
            f"valid_loss={val_loss:.5f}, "
            f"hamming@0.5={val_hl_05:.5f}, "
            f"seconds={epoch_seconds:.1f}"
        )
        print("val pred rates @0.5:", np.round(val_rates_05, 4).tolist())
        gpu_memory(f"After fold {fold} epoch {epoch} ")

        checkpoint_metric = val_hl_05 if CHECKPOINT_METRIC == "hamming_loss" else val_loss
        if checkpoint_metric < best_metric - 1e-6:
            best_metric = checkpoint_metric
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
            print(f"new best checkpoint by {CHECKPOINT_METRIC}: {best_metric:.5f}")
        else:
            bad_epochs += 1
            if bad_epochs > EARLY_STOPPING_PATIENCE:
                print("early stopping")
                break

        clear_memory()

    if SAVE_DIAGNOSTICS and epoch_logs:
        pd.DataFrame(epoch_logs).to_csv(f"diagnostics_transformer_fold_{fold}_epochs.csv", index=False)

    if best_state is not None:
        model.load_state_dict(best_state)
        del best_state
        clear_memory()

    val_probs = predict_transformer_encoded(model, train_encoded, indices=va_idx, batch_size=VALID_BATCH_SIZE)
    test_probs = predict_transformer_encoded(model, test_encoded, indices=None, batch_size=VALID_BATCH_SIZE)

    del model, ds_tr, loader_tr, optimizer, scheduler, scaler
    clear_memory()

    log_event("Transformer fold completed", fold=fold, seconds=round(time.time() - fold_t0, 1), best_valid_loss=round(best_loss, 5), best_metric=round(best_metric, 5))
    return np.clip(val_probs, 0, 1), np.clip(test_probs, 0, 1)


In [14]:
transformer_oof = np.zeros((len(train), N_LABELS), dtype=np.float32)
transformer_test = np.zeros((len(test), N_LABELS), dtype=np.float32)

if USE_TRANSFORMER:
    log_event("Transformer OOF training started")
    for fold in range(N_FOLDS):
        print("\nTRANSFORMER FOLD", fold)
        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]

        try:
            va_pred, te_pred = train_transformer_fold(fold, tr_idx, va_idx)
        except RuntimeError as e:
            clear_memory()
            if "out of memory" in str(e).lower() or "cuda" in str(e).lower():
                raise RuntimeError(
                    "Пойман CUDA OOM. Аварийные изменения без смены архитектуры: "
                    "MAX_LEN=160, VALID_BATCH_SIZE=8, TRAIN_BATCH_SIZE=2, GRAD_ACCUM_STEPS=4. "
                    "Исходная ошибка: " + repr(e)
                )
            raise

        transformer_oof[va_idx] = va_pred
        transformer_test += te_pred / N_FOLDS
        fold_hl_05 = hamming_loss(Y[va_idx], (va_pred >= 0.5).astype(int))
        log_event("Transformer OOF fold stored", fold=fold, hamming_at_05=round(fold_hl_05, 5))

    print("Transformer OOF hamming at 0.5:", hamming_loss(Y, (transformer_oof >= 0.5).astype(int)))
    transformer_rate_report = pd.DataFrame({
        "class": [f"class_{j}" for j in range(N_LABELS)],
        "true_rate": Y.mean(axis=0),
        "transformer_prob_mean": transformer_oof.mean(axis=0),
        "transformer_pred_rate_05": (transformer_oof >= 0.5).mean(axis=0),
    })
    display(transformer_rate_report)
    if SAVE_DIAGNOSTICS:
        transformer_rate_report.to_csv("diagnostics_transformer_oof_rates.csv", index=False)
else:
    transformer_oof[:] = 0.0
    transformer_test[:] = 0.0


[  10.13 min] Transformer OOF training started

TRANSFORMER FOLD 0
Before transformer fold 0 GPU memory: allocated=0.00 GB, reserved=0.00 GB
pos_weight: [1.184000015258789, 2.9210000038146973, 3.3559999465942383, 4.316999912261963, 5.25]


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cac

...

Transformer fold 1: trainable params 79,150,085 / 178,446,341
fold 1 epoch 1: train_loss=0.39907, valid_loss=0.29958, hamming@0.5=0.06704, seconds=590.8
val pred rates @0.5: [0.3722, 0.1839, 0.1326, 0.1205, 0.0251]
After fold 1 epoch 1 GPU memory: allocated=1.32 GB, reserved=2.13 GB
new best checkpoint by hamming_loss: 0.06704
fold 1 epoch 2: train_loss=0.28374, valid_loss=0.29383, hamming@0.5=0.05256, seconds=592.6
val pred rates @0.5: [0.4236, 0.1423, 0.1049, 0.0977, 0.0262]
After fold 1 epoch 2 GPU memory: allocated=1.32 GB, reserved=2.13 GB
new best checkpoint by hamming_loss: 0.05256
fold 1 epoch 3: train_loss=0.23353, valid_loss=0.30850, hamming@0.5=0.04850, seconds=617.1
val pred rates @0.5: [0.4327, 0.1385, 0.1013, 0.0656, 0.0241]
After fold 1 epoch 3 GPU memory: allocated=1.32 GB, reserved=2.13 GB
new best checkpoint by hamming_loss: 0.04850
fold 1 epoch 4: train_loss=0.19049, valid_loss=0.32158, hamming@0.5=0.04969, seconds=606.5
val pred rates @0.5: [0.4372, 0.1369, 0.1008, 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer fold 2: trainable params 79,150,085 / 178,446,341
fold 2 epoch 1: train_loss=0.38805, valid_loss=0.32745, hamming@0.5=0.05784, seconds=606.1
val pred rates @0.5: [0.4746, 0.1703, 0.0893, 0.0627, 0.0278]
After fold 2 epoch 1 GPU memory: allocated=1.32 GB, reserved=2.13 GB
new best checkpoint by hamming_loss: 0.05784
fold 2 epoch 2: train_loss=0.28369, valid_loss=0.31382, hamming@0.5=0.05109, seconds=606.8
val pred rates @0.5: [0.467, 0.1358, 0.0869, 0.074, 0.0243]
After fold 2 epoch 2 GPU memory: allocated=1.32 GB, reserved=2.13 GB
new best checkpoint by hamming_loss: 0.05109
fold 2 epoch 3: train_loss=0.22813, valid_loss=0.30650, hamming@0.5=0.04911, seconds=612.8
val pred rates @0.5: [0.4351, 0.1335, 0.1099, 0.081, 0.0187]
After fold 2 epoch 3 GPU memory: allocated=1.32 GB, reserved=2.13 GB
new best checkpoint by hamming_loss: 0.04911
fold 2 epoch 4: train_loss=0.18516, valid_loss=0.32460, hamming@0.5=0.04850, seconds=605.7
val pred rates @0.5: [0.4331, 0.1381, 0.0997, 0.0

,class,true_rate,transformer_prob_mean,transformer_pred_rate_05
0,class_0,0.427819,0.437025,0.430274
1,class_1,0.136279,0.150698,0.135980
2,class_2,0.109934,0.120479,0.098737
3,class_3,0.074427,0.087955,0.069517
4,class_4,0.032573,0.045801,0.024549


## 7. Meta-stacking

На этом этапе объединяются предсказания классических моделей, transformer-модели и дополнительные табличные признаки. Meta-модель учится на OOF-предсказаниях, поэтому она видит поведение базовых моделей на данных, которые не использовались для их обучения.

Цель stacking — не заменить базовые модели, а аккуратно скорректировать их ошибки и получить более устойчивые вероятности для каждого класса.


In [15]:
def build_meta_features(df, classical_probs, transformer_probs):
    df = df.reset_index(drop=True)
    classical_probs = np.asarray(classical_probs, dtype=np.float32)
    transformer_probs = np.asarray(transformer_probs, dtype=np.float32)
    avg_probs = 0.5 * classical_probs + 0.5 * transformer_probs
    diff_probs = transformer_probs - classical_probs
    prod_probs = transformer_probs * classical_probs

    prob_features = np.hstack([
        classical_probs,
        transformer_probs,
        avg_probs,
        diff_probs,
        np.abs(diff_probs),
        prod_probs,
    ])

    sorted_probs = np.sort(avg_probs, axis=1)
    row_stats = np.column_stack([
        avg_probs.max(axis=1),
        avg_probs.sum(axis=1),
        sorted_probs[:, -2],
        sorted_probs[:, -1] - sorted_probs[:, -2],
        (avg_probs > 0.35).sum(axis=1),
        (avg_probs > 0.50).sum(axis=1),
        np.log1p(df["clean_title_len"].values),
        np.log1p(df["clean_text_len"].values),
        np.log1p(df["bert_text_len"].values),
    ]).astype(np.float32)

    source_group = df["source_group"].fillna("other").astype(str)
    source_features = np.column_stack([
        (source_group == "novosti").astype(np.float32),
        (source_group == "svezhesti").astype(np.float32),
        (source_group == "other").astype(np.float32),
    ])

    dt = pd.to_datetime(df["publication_date"], errors="coerce")
    month = dt.dt.month.fillna(0).astype(np.float32).values.reshape(-1, 1) / 12.0

    return np.hstack([prob_features, row_stats, source_features, month]).astype(np.float32)

def fit_predict_meta_stacking(train_features, test_features, y, folds):
    meta_oof = np.zeros((len(train_features), N_LABELS), dtype=np.float32)
    meta_test = np.zeros((len(test_features), N_LABELS), dtype=np.float32)

    for fold in range(N_FOLDS):
        print("\nMETA FOLD", fold)
        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(train_features[tr_idx])
        X_va = scaler.transform(train_features[va_idx])
        X_te = scaler.transform(test_features)

        fold_test_parts = []

        for j in range(N_LABELS):
            y_tr = y[tr_idx, j]
            plain = LogisticRegression(
                C=2.0,
                solver="lbfgs",
                max_iter=1000,
                class_weight=None,
                random_state=SEED
            )
            balanced = LogisticRegression(
                C=0.8,
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
                random_state=SEED
            )

            plain.fit(X_tr, y_tr)
            balanced.fit(X_tr, y_tr)

            va_prob = 0.68 * plain.predict_proba(X_va)[:, 1] + 0.32 * balanced.predict_proba(X_va)[:, 1]
            te_prob = 0.68 * plain.predict_proba(X_te)[:, 1] + 0.32 * balanced.predict_proba(X_te)[:, 1]

            meta_oof[va_idx, j] = va_prob.astype(np.float32)
            fold_test_parts.append(te_prob.astype(np.float32))

        fold_test = np.column_stack(fold_test_parts)
        meta_test += fold_test / N_FOLDS

        fold_loss_05 = hamming_loss(y[va_idx], (meta_oof[va_idx] >= 0.5).astype(int))
        fold_rates = (meta_oof[va_idx] >= 0.5).mean(axis=0)
        log_event("Meta fold completed", fold=fold, hamming_at_05=round(fold_loss_05, 5))
        print("meta val pred rates @0.5:", np.round(fold_rates, 4).tolist())

    return np.clip(meta_oof, 0, 1), np.clip(meta_test, 0, 1)

if USE_META_STACKING and USE_CLASSICAL and USE_TRANSFORMER:
    log_event("Meta-stacking started")
    meta_train_features = build_meta_features(train, classical_oof, transformer_oof)
    meta_test_features = build_meta_features(test, classical_test, transformer_test)
    print("Meta feature shapes:", meta_train_features.shape, meta_test_features.shape)

    meta_oof, meta_test = fit_predict_meta_stacking(meta_train_features, meta_test_features, Y, folds)
    print("Meta OOF hamming at 0.5:", hamming_loss(Y, (meta_oof >= 0.5).astype(int)))

    meta_rate_report = pd.DataFrame({
        "class": [f"class_{j}" for j in range(N_LABELS)],
        "true_rate": Y.mean(axis=0),
        "meta_prob_mean": meta_oof.mean(axis=0),
        "meta_pred_rate_05": (meta_oof >= 0.5).mean(axis=0),
    })
    display(meta_rate_report)

    if SAVE_DIAGNOSTICS:
        meta_rate_report.to_csv("diagnostics_meta_oof_rates.csv", index=False)
else:
    meta_oof = np.zeros_like(transformer_oof, dtype=np.float32)
    meta_test = np.zeros_like(transformer_test, dtype=np.float32)


[ 133.39 min] Meta-stacking started
Meta feature shapes: (16701, 43) (4969, 43)

META FOLD 0
[ 133.49 min] Meta fold completed | fold=0, hamming_at_05=0.04846
meta val pred rates @0.5: [0.4245, 0.1302, 0.1013, 0.0726, 0.0264]

META FOLD 1
[ 133.60 min] Meta fold completed | fold=1, hamming_at_05=0.04807
meta val pred rates @0.5: [0.4315, 0.141, 0.1081, 0.0681, 0.0243]

META FOLD 2
[ 133.71 min] Meta fold completed | fold=2, hamming_at_05=0.04796
meta val pred rates @0.5: [0.4315, 0.1392, 0.1029, 0.0756, 0.0246]
Meta OOF hamming at 0.5: 0.048164780552062754


,class,true_rate,meta_prob_mean,meta_pred_rate_05
0,class_0,0.427819,0.433904,0.429136
1,class_1,0.136279,0.166014,0.136818
2,class_2,0.109934,0.153461,0.104126
3,class_3,0.074427,0.124706,0.072091
4,class_4,0.032573,0.110550,0.025088


## 8. Ансамбль и подбор порогов

После получения вероятностей подбираются индивидуальные пороги для каждого класса. Это особенно важно из-за дисбаланса: частые и редкие классы не должны декодироваться одним и тем же порогом.

Также отдельно учитывается вероятность объекта остаться без меток. В этой задаче много all-zero примеров, поэтому decoder должен не навязывать метку каждому тексту, а выбирать её только при достаточной уверенности модели.


In [16]:
def optimize_single_threshold(y_col, prob_col):
    best_t = 0.5
    best_loss = 10.0
    for t in np.linspace(0.03, 0.97, 189):
        pred_col = (prob_col >= t).astype(int)
        loss = np.mean(pred_col != y_col)
        if loss < best_loss:
            best_loss = loss
            best_t = float(t)
    return best_t, float(best_loss)

def optimize_thresholds(y_true, probs):
    n_cols = probs.shape[1]
    thresholds = np.zeros(n_cols, dtype=np.float32)
    for j in range(n_cols):
        thresholds[j], _ = optimize_single_threshold(y_true[:, j], probs[:, j])
    return thresholds

def apply_thresholds(probs, thresholds, zero_gate=None):
    pred = (probs >= thresholds.reshape(1, -1)).astype(int)
    if zero_gate is not None:
        pred[probs.max(axis=1) < zero_gate] = 0
    return pred

def optimize_zero_gate(y_true, probs, thresholds):
    best_gate = None
    best_loss = hamming_loss(y_true, apply_thresholds(probs, thresholds, None))
    for gate in np.linspace(0.05, 0.70, 131):
        pred = apply_thresholds(probs, thresholds, gate)
        loss = hamming_loss(y_true, pred)
        if loss < best_loss:
            best_loss = loss
            best_gate = float(gate)
    return best_gate, best_loss

def candidate_weight_grid(has_meta):
    if not has_meta:
        for wt in np.linspace(0.25, 0.95, 29):
            yield np.array([1.0 - wt, wt], dtype=np.float32)
    else:
        for wc in np.linspace(0.00, 0.70, 15):
            for wt in np.linspace(0.00, 0.85, 18):
                wm = 1.0 - wc - wt
                if wm < -1e-9 or wm > 0.95:
                    continue
                yield np.array([wc, wt, wm], dtype=np.float32)

def optimize_ensemble(y_true, classical_probs, transformer_probs, meta_probs=None):
    if meta_probs is None:
        source_names = ["classical", "transformer"]
        source_probs = [classical_probs, transformer_probs]
    else:
        source_names = ["classical", "transformer", "meta"]
        source_probs = [classical_probs, transformer_probs, meta_probs]

    source_weights = np.zeros((N_LABELS, len(source_names)), dtype=np.float32)
    mixed = np.zeros_like(transformer_probs, dtype=np.float32)

    for j in range(N_LABELS):
        best_loss = 10.0
        best_w = None
        best_col = None

        for w in candidate_weight_grid(meta_probs is not None):
            col = np.zeros(len(y_true), dtype=np.float32)
            for k, probs in enumerate(source_probs):
                col += w[k] * probs[:, j]
            _, loss = optimize_single_threshold(y_true[:, j], col)
            if loss < best_loss:
                best_loss = loss
                best_w = w.copy()
                best_col = col.copy()

        source_weights[j] = best_w
        mixed[:, j] = best_col

    thresholds = optimize_thresholds(y_true, mixed)
    zero_gate, loss_with_gate = optimize_zero_gate(y_true, mixed, thresholds)
    pred = apply_thresholds(mixed, thresholds, zero_gate)
    loss = hamming_loss(y_true, pred)

    return {
        "source_names": source_names,
        "source_weights": source_weights,
        "thresholds": thresholds,
        "zero_gate": zero_gate,
        "oof_probs": mixed,
        "oof_pred": pred,
        "oof_loss": loss,
        "oof_loss_with_gate": loss_with_gate
    }

def set_top_k_for_class(pred, probs, class_idx, k):
    pred[:, class_idx] = 0
    if k > 0:
        idx = np.argsort(-probs[:, class_idx])[:k]
        pred[idx, class_idx] = 1

def apply_global_rate_floor_cap(pred, probs, train_rates, oof_rates):
    pred = pred.copy()
    actions = []
    n = len(pred)

    low_targets = np.maximum(GLOBAL_RATE_FLOOR_FROM_OOF * oof_rates,
                             GLOBAL_RATE_FLOOR_FROM_TRAIN * train_rates)
    high_targets = np.minimum(0.95, GLOBAL_RATE_CAP_FROM_TRAIN * train_rates + 0.005)

    for j in range(N_LABELS):
        current = float(pred[:, j].mean())
        low = float(low_targets[j])
        high = float(high_targets[j])

        if current < low:
            k = max(1, int(round(low * n)))
            set_top_k_for_class(pred, probs, j, k)
            actions.append((j, "raised_global_floor", current, float(pred[:, j].mean()), low, high))
        elif current > high:
            k = max(0, int(round(high * n)))
            set_top_k_for_class(pred, probs, j, k)
            actions.append((j, "lowered_global_cap", current, float(pred[:, j].mean()), low, high))

    return pred, actions

def apply_multilabel_rescue(pred, probs, thresholds, min_multilabel_rate, allowed_classes=None):
    pred = pred.copy()
    if allowed_classes is None:
        allowed_classes = list(range(N_LABELS))

    n = len(pred)
    current_multi = int((pred.sum(axis=1) >= 2).sum())
    target_multi = int(round(min_multilabel_rate * n))
    need = target_multi - current_multi

    info = {
        "current_multi_rate": current_multi / max(n, 1),
        "target_multi_rate": target_multi / max(n, 1),
        "added_labels": 0,
        "candidate_count": 0,
    }

    if need <= 0:
        return pred, info

    candidates = []
    ratios_min = np.array([0.86, 0.74, 0.76, 0.74, 0.58], dtype=np.float32)

    one_label_rows = np.where(pred.sum(axis=1) == 1)[0]
    for i in one_label_rows:
        best_j = None
        best_score = -1e9
        for j in allowed_classes:
            if pred[i, j] == 1:
                continue
            if thresholds[j] <= 0:
                continue
            ratio = probs[i, j] / thresholds[j]
            if ratio < ratios_min[j]:
                continue
            # Отдаём приоритет редким/недопредсказанным классам.
            rare_bonus = [0.00, 0.04, 0.03, 0.06, 0.10][j]
            score = ratio + rare_bonus
            if score > best_score:
                best_score = score
                best_j = j
        if best_j is not None:
            candidates.append((best_score, i, best_j))

    info["candidate_count"] = len(candidates)
    if not candidates:
        return pred, info

    candidates.sort(reverse=True)
    added = 0
    for _, i, j in candidates[:need]:
        if pred[i, j] == 0:
            pred[i, j] = 1
            added += 1

    info["added_labels"] = added
    info["new_multi_rate"] = float((pred.sum(axis=1) >= 2).mean())
    return pred, info

log_event("Ensemble and threshold optimization started")

if USE_TRANSFORMER and USE_CLASSICAL and USE_META_STACKING:
    ens = optimize_ensemble(Y, classical_oof, transformer_oof, meta_oof)
elif USE_TRANSFORMER and USE_CLASSICAL:
    ens = optimize_ensemble(Y, classical_oof, transformer_oof, None)
elif USE_TRANSFORMER:
    thresholds = optimize_thresholds(Y, transformer_oof)
    zero_gate, _ = optimize_zero_gate(Y, transformer_oof, thresholds)
    ens = {
        "source_names": ["transformer"],
        "source_weights": np.ones((N_LABELS, 1), dtype=np.float32),
        "thresholds": thresholds,
        "zero_gate": zero_gate,
        "oof_probs": transformer_oof,
        "oof_pred": apply_thresholds(transformer_oof, thresholds, zero_gate),
        "oof_loss": hamming_loss(Y, apply_thresholds(transformer_oof, thresholds, zero_gate))
    }
else:
    thresholds = optimize_thresholds(Y, classical_oof)
    zero_gate, _ = optimize_zero_gate(Y, classical_oof, thresholds)
    ens = {
        "source_names": ["classical"],
        "source_weights": np.ones((N_LABELS, 1), dtype=np.float32),
        "thresholds": thresholds,
        "zero_gate": zero_gate,
        "oof_probs": classical_oof,
        "oof_pred": apply_thresholds(classical_oof, thresholds, zero_gate),
        "oof_loss": hamming_loss(Y, apply_thresholds(classical_oof, thresholds, zero_gate))
    }

print("OOF hamming_loss:", ens["oof_loss"])
print("Source names:", ens["source_names"])
print("Source weights by class:")
display(pd.DataFrame(ens["source_weights"], columns=ens["source_names"], index=[f"class_{j}" for j in range(N_LABELS)]))
print("Thresholds:", np.round(ens["thresholds"], 3).tolist())
print("Zero gate:", ens["zero_gate"])

precision, recall, f1, support = precision_recall_fscore_support(Y, ens["oof_pred"], average=None, zero_division=0)
per_class_errors = []
for j in range(N_LABELS):
    yj = Y[:, j]
    pj = ens["oof_pred"][:, j]
    row = {
        "class": f"class_{j}",
        "true_rate": float(yj.mean()),
        "oof_prob_mean": float(ens["oof_probs"][:, j].mean()),
        "oof_pred_rate": float(pj.mean()),
        "threshold": float(ens["thresholds"][j]),
        "class_hamming_loss": float(np.mean(yj != pj)),
        "false_positive_rate_cells": float(((yj == 0) & (pj == 1)).mean()),
        "false_negative_rate_cells": float(((yj == 1) & (pj == 0)).mean()),
        "precision": float(precision[j]),
        "recall": float(recall[j]),
        "f1": float(f1[j]),
        "support": int(support[j]),
    }
    for k, name in enumerate(ens["source_names"]):
        row[f"weight_{name}"] = float(ens["source_weights"][j, k])
    per_class_errors.append(row)

diagnostic = pd.DataFrame(per_class_errors)
print("OOF diagnostic by class:")
display(diagnostic)

pattern_report = pd.DataFrame({
    "true_pattern": pd.Series(["".join(map(str, row)) for row in Y]).value_counts(),
}).join(
    pd.DataFrame({"pred_pattern": pd.Series(["".join(map(str, row)) for row in ens["oof_pred"]]).value_counts()}),
    how="outer"
).fillna(0).astype(int).head(25)
print("Top target patterns, true vs OOF predicted:")
display(pattern_report)

true_all_zero_rate = float((Y.sum(axis=1) == 0).mean())
pred_all_zero_rate = float((ens["oof_pred"].sum(axis=1) == 0).mean())
true_multilabel_rate = float((Y.sum(axis=1) >= 2).mean())
pred_multilabel_rate = float((ens["oof_pred"].sum(axis=1) >= 2).mean())
print("OOF all-zero true:", true_all_zero_rate)
print("OOF all-zero pred:", pred_all_zero_rate)
print("OOF multilabel true:", true_multilabel_rate)
print("OOF multilabel pred:", pred_multilabel_rate)

# Диагностируем два осторожных постпроцессинга на OOF:
# 1) floor/cap по rates, 2) добавление второй метки только для near-threshold строк.
oof_floor_pred, oof_floor_actions = apply_global_rate_floor_cap(
    ens["oof_pred"], ens["oof_probs"], Y.mean(axis=0), ens["oof_pred"].mean(axis=0)
)
oof_floor_loss = hamming_loss(Y, oof_floor_pred)

target_multi_rate = MIN_MULTILABEL_RATE_FROM_TRAIN * true_multilabel_rate
oof_rescue_pred, oof_rescue_info = apply_multilabel_rescue(
    oof_floor_pred, ens["oof_probs"], ens["thresholds"], min_multilabel_rate=target_multi_rate
)
oof_rescue_loss = hamming_loss(Y, oof_rescue_pred)

APPLY_MULTILABEL_RESCUE_ON_TEST = bool(
    USE_MULTILABEL_RESCUE and (oof_rescue_loss <= ens["oof_loss"] + MAX_MULTILABEL_RESCUE_OOF_DELTA)
)

postprocess_oof_report = pd.DataFrame([
    {
        "variant": "base",
        "hamming_loss": ens["oof_loss"],
        "all_zero_rate": pred_all_zero_rate,
        "multilabel_rate": pred_multilabel_rate,
    },
    {
        "variant": "rate_floor_cap",
        "hamming_loss": oof_floor_loss,
        "all_zero_rate": float((oof_floor_pred.sum(axis=1) == 0).mean()),
        "multilabel_rate": float((oof_floor_pred.sum(axis=1) >= 2).mean()),
    },
    {
        "variant": "rate_floor_cap_plus_multilabel_rescue",
        "hamming_loss": oof_rescue_loss,
        "all_zero_rate": float((oof_rescue_pred.sum(axis=1) == 0).mean()),
        "multilabel_rate": float((oof_rescue_pred.sum(axis=1) >= 2).mean()),
    },
])
print("OOF postprocess variants:")
display(postprocess_oof_report)
print("OOF rate floor actions:", oof_floor_actions)
print("OOF multilabel rescue info:", oof_rescue_info)
print("Apply multilabel rescue on test:", APPLY_MULTILABEL_RESCUE_ON_TEST)

if SAVE_DIAGNOSTICS:
    diagnostic.to_csv("diagnostics_oof_per_class.csv", index=False)
    pattern_report.to_csv("diagnostics_oof_patterns.csv")
    postprocess_oof_report.to_csv("diagnostics_oof_postprocess_variants.csv", index=False)
    pd.DataFrame(ens["source_weights"], columns=ens["source_names"], index=[f"class_{j}" for j in range(N_LABELS)]).to_csv("diagnostics_ensemble_weights.csv")

log_event("Ensemble optimized", oof_hamming_loss=round(float(ens["oof_loss"]), 6))


[ 133.71 min] Ensemble and threshold optimization started
OOF hamming_loss: 0.04548230644871565
Source names: ['classical', 'transformer', 'meta']
Source weights by class:


,classical,transformer,meta
class_0,0.35,0.65,1.110223e-16
class_1,0.50,0.50,5.551115e-17
class_2,0.45,0.55,1.110223e-16
class_3,0.40,0.60,1.110223e-16
class_4,0.15,0.50,3.500000e-01


Thresholds: [0.550000011920929, 0.49000000953674316, 0.5299999713897705, 0.5249999761581421, 0.6299999952316284]
Zero gate: None
OOF diagnostic by class:


,class,true_rate,oof_prob_mean,oof_pred_rate,threshold,class_hamming_loss,false_positive_rate_cells,false_negative_rate_cells,precision,recall,f1,support,weight_classical,weight_transformer,weight_meta
0,class_0,0.427819,0.433345,0.418957,0.550,0.083588,0.037363,0.046225,0.910819,0.891952,0.901287,7145,0.35,0.65,1.110223e-16
1,class_1,0.136279,0.149657,0.132208,0.490,0.043949,0.019939,0.024011,0.849185,0.823814,0.836307,2276,0.50,0.50,5.551115e-17
2,class_2,0.109934,0.124028,0.093467,0.530,0.044967,0.014251,0.030717,0.847534,0.720588,0.778923,1836,0.45,0.55,1.110223e-16
3,class_3,0.074427,0.086640,0.064128,0.525,0.037842,0.013772,0.024070,0.785247,0.676589,0.726880,1243,0.40,0.60,1.110223e-16
4,class_4,0.032573,0.067137,0.022334,0.630,0.017065,0.003413,0.013652,0.847185,0.580882,0.689204,544,0.15,0.50,3.500000e-01


Top target patterns, true vs OOF predicted:


,true_pattern,pred_pattern
00000,5643,5922
00001,303,308
00010,516,540
00011,11,1
00100,1455,1447
00101,90,43
00110,60,11
00111,5,0
01000,1169,1199
01001,23,3


OOF all-zero true: 0.3378839590443686
OOF all-zero pred: 0.3545895455361954
OOF multilabel true: 0.11071193341716065
OOF multilabel pred: 0.08113286629543141
OOF postprocess variants:


,variant,hamming_loss,all_zero_rate,multilabel_rate
0,base,0.045482,0.35459,0.081133
1,rate_floor_cap,0.045578,0.35435,0.081372
2,rate_floor_cap_plus_multilabel_rescue,0.045578,0.35435,0.081372


OOF rate floor actions: [(4, 'raised_global_floor', 0.022333991976528352, 0.022813005209268905, 0.022801029490151167, 0.041481647960840314)]
OOF multilabel rescue info: {'current_multi_rate': 0.08137237291180169, 'target_multi_rate': 0.07969582659720975, 'added_labels': 0, 'candidate_count': 0}
Apply multilabel rescue on test: True
[ 133.95 min] Ensemble optimized | oof_hamming_loss=0.045482


## 9. Прогноз на test

Для тестовой выборки применяются те же источники вероятностей и те же правила декодирования, которые были выбраны на OOF-валидации. Основной прогноз строится консервативно: он старается не завышать число положительных меток и сохраняет баланс между частыми и редкими классами.


In [17]:
thresholds = ens["thresholds"].copy()
zero_gate = ens["zero_gate"]

test_prob_sources = {
    "classical": classical_test,
    "transformer": transformer_test,
    "meta": meta_test,
}

test_probs = np.zeros((len(test), N_LABELS), dtype=np.float32)
for k, name in enumerate(ens["source_names"]):
    test_probs += ens["source_weights"][:, k].reshape(1, -1) * test_prob_sources[name]

test_probs = np.clip(test_probs, 0, 1)
test_pred_before_guard = apply_thresholds(test_probs, thresholds, zero_gate)
test_pred_after_rate = test_pred_before_guard.copy()
test_pred_after_source = test_pred_before_guard.copy()
test_pred_after_rescue = test_pred_before_guard.copy()

if USE_RATE_GUARD:
    test_pred_after_rate, rate_adjustments = apply_global_rate_floor_cap(
        test_pred_before_guard,
        test_probs,
        train_rates=Y.mean(axis=0),
        oof_rates=ens["oof_pred"].mean(axis=0),
    )
else:
    test_pred_after_rate = test_pred_before_guard.copy()
    rate_adjustments = []

def apply_known_source_guard(pred, probs, test_df, train_df, y):
    pred = pred.copy()
    adjusted = []
    train_tmp = train_df.copy()
    for j in range(N_LABELS):
        train_tmp[f"class_{j}"] = y[:, j]

    source_rates = train_tmp.groupby("source_group")[[f"class_{j}" for j in range(N_LABELS)]].mean()

    for source_group in ["novosti", "svezhesti"]:
        mask = test_df["source_group"].values == source_group
        n = int(mask.sum())
        if n < 50 or source_group not in source_rates.index:
            continue

        for j in range(N_LABELS):
            target = float(source_rates.loc[source_group, f"class_{j}"])
            if target <= 0:
                continue

            current_rate = float(pred[mask, j].mean())
            if j == 4:
                low_mult = 0.68
            elif j in {1, 3}:
                low_mult = 0.72
            else:
                low_mult = 0.66

            low = low_mult * target
            high = min(0.95, 1.75 * target + 0.01)

            if current_rate < low:
                k = max(1, int(round(low * n)))
                local_indices = np.where(mask)[0]
                chosen_local = local_indices[np.argsort(-probs[mask, j])[:k]]
                pred[local_indices, j] = 0
                pred[chosen_local, j] = 1
                adjusted.append((j, f"raised_source_{source_group}", current_rate, float(pred[mask, j].mean()), low, high, n))
            elif current_rate > high:
                k = max(0, int(round(high * n)))
                local_indices = np.where(mask)[0]
                chosen_local = local_indices[np.argsort(-probs[mask, j])[:k]]
                pred[local_indices, j] = 0
                if k > 0:
                    pred[chosen_local, j] = 1
                adjusted.append((j, f"lowered_source_{source_group}", current_rate, float(pred[mask, j].mean()), low, high, n))
    return pred, adjusted

def apply_unknown_source_guard(pred, probs, test_df):
    pred = pred.copy()
    adjusted = []
    mask = test_df["source_group"].values == "other"
    n = int(mask.sum())
    if n < 50:
        return pred, adjusted

    # После прогона 0.04176 unknown sources были слишком осторожными по class_4.
    # Поднимаем только минимально и только top-probability строки.
    unknown_floors = {
        1: max(0.090, 0.68 * float(Y[:, 1].mean())),
        4: max(0.018, 0.55 * float(Y[:, 4].mean())),
    }

    local_indices = np.where(mask)[0]
    for j, low in unknown_floors.items():
        current_rate = float(pred[mask, j].mean())
        if current_rate < low:
            k = max(1, int(round(low * n)))
            chosen_local = local_indices[np.argsort(-probs[mask, j])[:k]]
            pred[local_indices, j] = 0
            pred[chosen_local, j] = 1
            adjusted.append((j, "raised_source_other", current_rate, float(pred[mask, j].mean()), low, np.nan, n))
    return pred, adjusted

source_adjustments = []
unknown_source_adjustments = []
if USE_SOURCE_AWARE_GUARD:
    test_pred_after_source, source_adjustments = apply_known_source_guard(test_pred_after_rate, test_probs, test, train, Y)
    test_pred_after_source, unknown_source_adjustments = apply_unknown_source_guard(test_pred_after_source, test_probs, test)
else:
    test_pred_after_source = test_pred_after_rate.copy()

true_multilabel_rate = float((Y.sum(axis=1) >= 2).mean())
target_multilabel_rate = MIN_MULTILABEL_RATE_FROM_TRAIN * true_multilabel_rate

if APPLY_MULTILABEL_RESCUE_ON_TEST:
    test_pred_after_rescue, multilabel_rescue_info_test = apply_multilabel_rescue(
        test_pred_after_source,
        test_probs,
        thresholds,
        min_multilabel_rate=target_multilabel_rate,
    )
else:
    test_pred_after_rescue = test_pred_after_source.copy()
    multilabel_rescue_info_test = {"skipped": True, "reason": "OOF did not allow rescue or disabled"}

test_pred = test_pred_after_rescue.copy()

def pred_rate_row(name, pred):
    row = {
        "stage": name,
        "all_zero_rate": float((pred.sum(axis=1) == 0).mean()),
        "multilabel_rate": float((pred.sum(axis=1) >= 2).mean()),
    }
    for j in range(N_LABELS):
        row[f"class_{j}_rate"] = float(pred[:, j].mean())
    return row

postprocess_test_report = pd.DataFrame([
    pred_rate_row("before_guard", test_pred_before_guard),
    pred_rate_row("after_global_rate_guard", test_pred_after_rate),
    pred_rate_row("after_source_guard", test_pred_after_source),
    pred_rate_row("final_after_multilabel_rescue", test_pred),
])

print("Rate adjustments:", rate_adjustments)
print("Source adjustments:", source_adjustments)
print("Unknown source adjustments:", unknown_source_adjustments)
print("Multilabel rescue info test:", multilabel_rescue_info_test)
print("Test postprocess report:")
display(postprocess_test_report)

prob_quantiles = []
for j in range(N_LABELS):
    qs = np.quantile(test_probs[:, j], [0.01, 0.05, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
    prob_quantiles.append({
        "class": f"class_{j}",
        "q01": qs[0], "q05": qs[1], "q25": qs[2], "q50": qs[3],
        "q75": qs[4], "q90": qs[5], "q95": qs[6], "q99": qs[7],
        "threshold": thresholds[j],
        "test_pred_rate": test_pred[:, j].mean(),
        "train_true_rate": Y[:, j].mean(),
        "oof_pred_rate": ens["oof_pred"][:, j].mean(),
        "oof_false_negative_rate": float(((Y[:, j] == 1) & (ens["oof_pred"][:, j] == 0)).mean()),
        "oof_false_positive_rate": float(((Y[:, j] == 0) & (ens["oof_pred"][:, j] == 1)).mean()),
    })
prob_quantiles = pd.DataFrame(prob_quantiles)
print("Test probability quantiles:")
display(prob_quantiles)

source_test_report = test.assign(**{f"pred_class_{j}": test_pred[:, j] for j in range(N_LABELS)}).groupby("source")[[f"pred_class_{j}" for j in range(N_LABELS)]].mean()
print("Test predicted rates by source:")
display(source_test_report)

source_group_test_report = test.assign(**{f"pred_class_{j}": test_pred[:, j] for j in range(N_LABELS)}).groupby("source_group")[[f"pred_class_{j}" for j in range(N_LABELS)]].mean()
print("Test predicted rates by source_group:")
display(source_group_test_report)

pattern_counts_final = pd.Series(["".join(map(str, row)) for row in test_pred]).value_counts().head(30).to_frame("count")
print("Final test pattern counts:")
display(pattern_counts_final)

if SAVE_DIAGNOSTICS:
    prob_quantiles.to_csv("diagnostics_test_probability_quantiles.csv", index=False)
    postprocess_test_report.to_csv("diagnostics_test_postprocess_stages.csv", index=False)
    source_test_report.to_csv("diagnostics_test_pred_rates_by_source.csv")
    source_group_test_report.to_csv("diagnostics_test_pred_rates_by_source_group.csv")
    pattern_counts_final.to_csv("diagnostics_test_pattern_counts_final.csv")
    pd.DataFrame(rate_adjustments, columns=["class", "action", "old_rate", "new_rate", "low_guard", "high_guard"]).to_csv("diagnostics_rate_guard_adjustments.csv", index=False)
    pd.DataFrame(source_adjustments, columns=["class", "action", "old_rate", "new_rate", "low_guard", "high_guard", "n_source"]).to_csv("diagnostics_source_guard_adjustments.csv", index=False)
    pd.DataFrame(unknown_source_adjustments, columns=["class", "action", "old_rate", "new_rate", "low_guard", "high_guard", "n_source"]).to_csv("diagnostics_unknown_source_guard_adjustments.csv", index=False)

log_event("Test inference postprocessing completed")


Rate adjustments: [(1, 'raised_global_floor', 0.1054538136445965, 0.11893741195411552, 0.11898688386067767, 0.1480932214521204), (4, 'raised_global_floor', 0.017307305292815454, 0.022740994163815656, 0.022801029490151167, 0.041481647960840314)]
Source adjustments: [(4, 'raised_source_novosti', 0.004008016032064128, 0.009519038076152305, 0.009380045458108003, 0.03413982287013089, 1996)]
Unknown source adjustments: []
Multilabel rescue info test: {'current_multi_rate': 0.08070034212115114, 'target_multi_rate': 0.07969410344133629, 'added_labels': 0, 'candidate_count': 0}
Test postprocess report:


,stage,all_zero_rate,multilabel_rate,class_0_rate,class_1_rate,class_2_rate,class_3_rate,class_4_rate
0,before_guard,0.326826,0.068022,0.428456,0.105454,0.119541,0.074663,0.017307
1,after_global_rate_guard,0.320386,0.078889,0.428456,0.118937,0.119541,0.074663,0.022741
2,after_source_guard,0.319984,0.080700,0.428456,0.118937,0.119541,0.074663,0.024955
3,final_after_multilabel_rescue,0.319984,0.080700,0.428456,0.118937,0.119541,0.074663,0.024955


Test probability quantiles:


,class,q01,q05,q25,q50,q75,q90,q95,q99,threshold,test_pred_rate,train_true_rate,oof_pred_rate,oof_false_negative_rate,oof_false_positive_rate
0,class_0,0.007867,0.010912,0.030590,0.264928,0.927711,0.987922,0.992522,0.994898,0.550,0.428456,0.427819,0.418957,0.046225,0.037363
1,class_1,0.005963,0.007803,0.012238,0.019142,0.052924,0.535886,0.911135,0.989291,0.490,0.118937,0.136279,0.132208,0.024011,0.019939
2,class_2,0.008503,0.011538,0.017756,0.030778,0.104485,0.682280,0.941744,0.985702,0.530,0.119541,0.109934,0.093467,0.030717,0.014251
3,class_3,0.007883,0.009981,0.013663,0.019509,0.039676,0.307724,0.714357,0.932842,0.525,0.074663,0.074427,0.064128,0.024070,0.013772
4,class_4,0.024157,0.030903,0.040088,0.045086,0.052579,0.064351,0.080675,0.856887,0.630,0.024955,0.032573,0.022334,0.013652,0.003413


Test predicted rates by source:


...

Test predicted rates by source_group:


,pred_class_0,pred_class_1,pred_class_2,pred_class_3,pred_class_4
source_group,,,,,
novosti,0.404810,0.106212,0.133768,0.086673,0.009519
other,0.441164,0.130977,0.113514,0.070270,0.022037
svezhesti,0.457746,0.112676,0.095070,0.051056,0.091549


Final test pattern counts:


,count
10000,1849
00000,1590
00100,545
01000,321
00010,179
11000,157
00001,84
01010,79
10010,75
11010,27


[ 133.95 min] Test inference postprocessing completed


## 10. Финальный submission

Финальный файл формируется в формате соревнования: один `id` и один список из пяти бинарных меток для каждого объекта.

В качестве итогового используется консервативный вариант прогноза, который показал лучший результат среди проверенных вариантов. На выходе сохраняется только один файл — `sample_submission.csv`.


In [18]:
def target_to_string(row):
    return "[" + ",".join(str(int(x)) for x in row) + "]"

def make_submission_from_pred(pred, filename):
    sub = pd.DataFrame({
        "id": sample_submission["id"].values,
        "target": [target_to_string(row) for row in pred]
    })

    if sub.shape[0] != sample_submission.shape[0]:
        raise ValueError("Размер submission не совпадает с sample_submission")

    if sub["target"].isna().any():
        raise ValueError("В submission есть NaN")

    bad_format = ~sub["target"].str.match(r"^\[[01],[01],[01],[01],[01]\]$")
    if bad_format.any():
        raise ValueError("Некорректный формат target в submission")

    sub.to_csv(filename, index=False)
    return sub

submission = make_submission_from_pred(test_pred_before_guard, "sample_submission.csv")

print("Saved sample_submission.csv")
display(submission.head())
display(submission["target"].value_counts().head(25).to_frame("count"))

log_event("Submission saved", rows=len(submission))


Saved sample_submission.csv


,id,target
0,16701,"[0,0,0,0,0]"
1,16702,"[0,0,0,0,0]"
2,16703,"[0,0,0,0,0]"
3,16704,"[0,0,1,0,0]"
4,16705,"[0,0,0,0,0]"


,count
target,
"[1,0,0,0,0]",1886
"[0,0,0,0,0]",1624
"[0,0,1,0,0]",557
"[0,1,0,0,0]",306
"[0,0,0,1,0]",189
"[1,1,0,0,0]",126
"[1,0,0,1,0]",83
"[0,1,0,1,0]",70
"[0,0,0,0,1]",69


[ 133.95 min] Submission saved | rows=4969


## 11. Проверка итогового результата

В конце фиксируются основные характеристики решения: локальная OOF-оценка, доли предсказанных классов и распределение количества меток на объект. Эта проверка нужна, чтобы убедиться, что итоговый файл не содержит пропусков, имеет правильный формат и не даёт резких перекосов по классам.


In [19]:
summary = {
    "local_oof_hamming_loss": float(ens["oof_loss"]),
    "oof_rate_floor_loss": float(oof_floor_loss),
    "oof_rescue_loss": float(oof_rescue_loss),
    "apply_multilabel_rescue_on_test": bool(APPLY_MULTILABEL_RESCUE_ON_TEST),
    "thresholds": [float(x) for x in ens["thresholds"]],
    "zero_gate": None if ens["zero_gate"] is None else float(ens["zero_gate"]),
    "source_names": ens["source_names"],
    "source_weights_by_class": ens["source_weights"].round(4).tolist(),
    "train_positive_rates": [float(x) for x in Y.mean(axis=0)],
    "oof_pred_positive_rates": [float(x) for x in ens["oof_pred"].mean(axis=0)],
    "test_pred_positive_rates": [float(x) for x in test_pred.mean(axis=0)],
    "train_all_zero_rate": float((Y.sum(axis=1) == 0).mean()),
    "oof_pred_all_zero_rate": float((ens["oof_pred"].sum(axis=1) == 0).mean()),
    "test_pred_all_zero_rate": float((test_pred.sum(axis=1) == 0).mean()),
    "train_multilabel_rate": float((Y.sum(axis=1) >= 2).mean()),
    "oof_pred_multilabel_rate": float((ens["oof_pred"].sum(axis=1) >= 2).mean()),
    "test_pred_multilabel_rate": float((test_pred.sum(axis=1) >= 2).mean()),
    "total_runtime_min_so_far": round((time.time() - RUN_STARTED_AT) / 60, 2),
}

summary_df = pd.Series(summary).to_frame("value")
display(summary_df)

print("Что смотреть после Kaggle submit:")
print("1) public hamming_loss vs local_oof_hamming_loss. В прошлом запуске public=0.04176, local OOF=0.04524, значит public был лучше локальной оценки.")
print("2) diagnostics_oof_per_class.csv: если false_negative_rate снова высокий у class_1/class_3/class_4, нужно ещё мягче снижать пороги или усиливать rescue.")
print("3) diagnostics_oof_patterns.csv и diagnostics_test_pattern_counts_final.csv: проверяем, не завышены ли [0,0,0,0,0] и одиночные классы.")
print("4) diagnostics_test_postprocess_stages.csv: показывает, что изменили global/source/rescue guards.")
print("5) diagnostics_test_probability_quantiles.csv: если threshold выше q95/q99, класс почти не будет предсказываться.")
print("6) diagnostics_test_predictions_debug.csv: строки около threshold можно использовать для анализа, но не для ручной подгонки target.")

,value
local_oof_hamming_loss,0.045482
oof_rate_floor_loss,0.045578
oof_rescue_loss,0.045578
apply_multilabel_rescue_on_test,True
thresholds,"[0.550000011920929, 0.49000000953674316, 0.529..."
zero_gate,None
source_names,"[classical, transformer, meta]"
source_weights_by_class,"[[0.3499999940395355, 0.6499999761581421, 0.0]..."
train_positive_rates,"[0.4278186934914077, 0.13627926471468774, 0.10..."
oof_pred_positive_rates,"[0.41895694868570743, 0.13220765223639303, 0.0..."


Что смотреть после Kaggle submit:
1) public hamming_loss vs local_oof_hamming_loss. В прошлом запуске public=0.04176, local OOF=0.04524, значит public был лучше локальной оценки.
2) diagnostics_oof_per_class.csv: если false_negative_rate снова высокий у class_1/class_3/class_4, нужно ещё мягче снижать пороги или усиливать rescue.
3) diagnostics_oof_patterns.csv и diagnostics_test_pattern_counts_final.csv: проверяем, не завышены ли [0,0,0,0,0] и одиночные классы.
4) diagnostics_test_postprocess_stages.csv: показывает, что изменили global/source/rescue guards.
5) diagnostics_test_probability_quantiles.csv: если threshold выше q95/q99, класс почти не будет предсказываться.
6) diagnostics_test_predictions_debug.csv: строки около threshold можно использовать для анализа, но не для ручной подгонки target.
